# Run Qwen2.5-32B-Instruct (4-bit)

Loads a 32B-parameter instruction-tuned model in 4-bit quantization so it fits on a single 48GB GPU (RTX A6000).

- Model: `unsloth/Qwen2.5-32B-Instruct-bnb-4bit` (pre-quantized, ~20GB download instead of ~65GB full precision)
- Quantization: bitsandbytes NF4, computed in bfloat16
- Model weights are cached under `/workspace/.cache/huggingface` (persistent volume) so they survive pod restarts


In [1]:
%pip install -q -U transformers accelerate bitsandbytes

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

# Keep the (large) model cache on the persistent volume, not the ephemeral container disk
os.environ["HF_HOME"] = "/workspace/.cache/huggingface"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "unsloth/Qwen2.5-32B-Instruct-bnb-4bit"

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

CUDA available: True
GPU: NVIDIA RTX A6000


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.eval()
print("Model loaded. Memory footprint (GB):", model.get_memory_footprint() / 1e9)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Model loaded. Memory footprint (GB): 18.719320576


In [4]:
def chat(prompt: str, system: str = "You are a helpful assistant.", max_new_tokens: int = 512) -> str:
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [5]:
response = chat("Explain the difference between a RunPod container disk and a network volume, in 3 sentences.")
print(response)

A RunPod container disk is local storage attached to a specific container, providing fast access but limited in size and persistence. A network volume, on the other hand, is a shared storage solution that can be mounted to multiple containers simultaneously, offering greater flexibility and data durability across different instances. While the container disk is ephemeral and data may be lost if the container is deleted, a network volume persists independently of the container's lifecycle.
